In [18]:
import xml.etree.ElementTree as ET
import requests

In [19]:
def get_response_content(year: int) -> bytes:
    url = (
        "https://home.treasury.gov/resource-center/"
        "data-chart-center/interest-rates/pages/xml"
    )

    params = {
        "data": "daily_treasury_yield_curve",
        "field_tdr_date_value": str(year),
    }
    try:
        response = requests.get(
            url,
            params=params,
            timeout=30,
        )
    except Exception as e:
        print(f"요청 실패: {e}")
        raise
    else:
        print(f"{year} 요청 성공")
        return response.content

In [20]:
def get_properties_child_tags(response_content: bytes) -> set[str]:
    root = ET.fromstring(response_content)

    namespace = {
        "atom": "http://www.w3.org/2005/Atom",
        "m": "http://schemas.microsoft.com/ado/2007/08/dataservices/metadata",
    }

    properties_list = root.findall(
        "atom:entry/atom:content/m:properties",
        namespace,
    )

    tags = {
        child.tag.split("}")[-1]
        for properties in properties_list
        for child in properties
    }

    print(f"파싱 성공: {tags}\n길이: {len(tags)}\n")
    return tags

In [21]:
all_tags = {}
for year in list(range(1990,2026)):
    content = get_response_content(year)
    tags = get_properties_child_tags(content)
    all_tags[year] = tags

print(tags)

1990 요청 성공
파싱 성공: {'BC_3YEAR', 'BC_3MONTH', 'BC_5YEAR', 'Id', 'BC_2YEAR', 'BC_1YEAR', 'NEW_DATE', 'BC_30YEAR', 'BC_30YEARDISPLAY', 'BC_7YEAR', 'BC_10YEAR', 'BC_6MONTH'}
길이: 12

1991 요청 성공
파싱 성공: {'BC_3YEAR', 'BC_3MONTH', 'BC_5YEAR', 'Id', 'BC_2YEAR', 'BC_1YEAR', 'NEW_DATE', 'BC_30YEAR', 'BC_30YEARDISPLAY', 'BC_7YEAR', 'BC_10YEAR', 'BC_6MONTH'}
길이: 12

1992 요청 성공
파싱 성공: {'BC_3YEAR', 'BC_3MONTH', 'BC_5YEAR', 'Id', 'BC_2YEAR', 'BC_1YEAR', 'NEW_DATE', 'BC_30YEAR', 'BC_30YEARDISPLAY', 'BC_7YEAR', 'BC_10YEAR', 'BC_6MONTH'}
길이: 12

1993 요청 성공
파싱 성공: {'BC_3YEAR', 'BC_3MONTH', 'BC_5YEAR', 'Id', 'BC_2YEAR', 'BC_1YEAR', 'NEW_DATE', 'BC_30YEAR', 'BC_20YEAR', 'BC_30YEARDISPLAY', 'BC_7YEAR', 'BC_10YEAR', 'BC_6MONTH'}
길이: 13

1994 요청 성공
파싱 성공: {'BC_3YEAR', 'BC_3MONTH', 'BC_5YEAR', 'Id', 'BC_2YEAR', 'BC_1YEAR', 'NEW_DATE', 'BC_30YEAR', 'BC_20YEAR', 'BC_30YEARDISPLAY', 'BC_7YEAR', 'BC_10YEAR', 'BC_6MONTH'}
길이: 13

1995 요청 성공
파싱 성공: {'BC_3YEAR', 'BC_3MONTH', 'BC_5YEAR', 'Id', 'BC_2YEAR', 'BC_1YEAR', 'NE

### Daily Treasury Par Yield Curve Rates 데이터 변화 확인
- [[extract_data.md]]의 주석에 맞는지 연도별로 확인한 결과
- 주석의 정책과 필드 변화가 일치함.
- 다만, 연도별 변화를 확인했기에 일별 변화는 아직 검증이 되지 않음.
의의: 큰 틀에서 주석과 실제 데이터 필드의 변화가 일치한 것을 확인함(1차 검증 완료)

### 정확한 요일별 검증을 위해 확인이 필요한 값들
검증할 항목	확인할 날짜
20년물 재개	1993-09-30, 1993-10-01
30년물 중단	2002-02-17, 2002-02-18
30년물 재개	2006-02-08, 2006-02-09
2개월물 도입	2018-10-15, 2018-10-16
4개월물 도입	2022-10-18, 2022-10-19
1.5개월물 도입	2025-02-17, 2025-02-18
